# Customer churn prediction — notebook walkthrough

**Dataset:** real historical US Cell2Cell telecom customers (51,047 rows in the verified local experiment).

This notebook provides the traditional data-science flow: question → exploration → preparation → model comparison → evaluation → interpretation. The reusable, tested implementation lives in `churn/train.py`. **This notebook is a later explanatory companion, not evidence that development originally started in a notebook.**

**Data rules:** no artificial customer rows, no silent fallback. Download the real dataset with `python -m churn.download` first. Keep customer-level CSVs and saved models local.

The published benchmark is in [docs/BENCHMARK.md](../docs/BENCHMARK.md), from a user-run experiment; notebook outputs are deliberately cleared.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "churn").is_dir(): ROOT = ROOT.parent
assert (ROOT / "churn").is_dir(), "Open Jupyter from project root or notebooks/"
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from churn.train import load_real_data, run, SEED
DATA = ROOT / "data" / "raw" / "cell2celltrain.csv"
X, y, sha = load_real_data(DATA)
print("Rows:",len(X),"Features:",X.shape[1],"SHA-256:",sha)
display(X.head())

## 1. Explore the target distribution and data quality
The true churn prevalence is the relevant baseline for average precision.

In [ ]:
print("Churn counts and percentages:")
display(pd.DataFrame({"count":y.value_counts().sort_index(),"percent":(100*y.value_counts(normalize=True).sort_index()).round(2)}))
display(X.isna().mean().sort_values(ascending=False).head(15).rename("missing_fraction").to_frame())
fig, ax = plt.subplots(figsize=(5,3))
y.map({0:"Retained",1:"Churned"}).value_counts().plot.bar(ax=ax)
ax.set_title("Real Cell2Cell customer outcome counts")
plt.tight_layout(); plt.show()

## 2. Explore numeric customer features
Plots are descriptive only. Never infer causality from these observational associations.

In [ ]:
num_cols = X.select_dtypes(include="number").columns.tolist()
display(X[num_cols].describe().T.head(20))
if num_cols:
    ax = X[num_cols[:min(4,len(num_cols))]].hist(figsize=(11,7),bins=25)
    plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()

## 3. Train and compare models
The production-oriented pipeline handles stratified 60/20/20 splitting, train-only imputation and encoding, validation average-precision selection, validation-only threshold optimization, and one frozen test evaluation. Training here writes only local files in `artifacts/`.

In [ ]:
from churn.train import run
results = run(DATA, ROOT / "artifacts")
comparison = pd.DataFrame(results["validation"]).T
display(comparison[["average_precision","roc_auc","precision","recall","f1"]].sort_values("average_precision",ascending=False))
print("Winner:",results["selected_model"],"threshold:",round(results["threshold"],3))

## 4. Held-out test evaluation and error analysis
These test results must not drive any further selection.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import precision_recall_curve, roc_curve
out = pd.read_csv(ROOT / "artifacts" / "test_predictions.csv")
display(pd.DataFrame([results["test"]]).drop(columns="confusion_matrix").T.rename(columns={0:"test"}))
from sklearn.metrics import confusion_matrix
fig,axes=plt.subplots(1,2,figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(out["actual_churn"],out["predicted_churn"],ax=axes[0],colorbar=False)
PrecisionRecallDisplay.from_predictions(out["actual_churn"],out["churn_probability"],ax=axes[1])
plt.tight_layout();plt.show()

## 5. Limitations and ML engineering transition
- Real historical US telecom records are not necessarily representative of today's US customers.
- Random stratified holdout is not a time-forward or independent-operator validation.
- High recall comes with many false alerts; consider capacity and intervention cost before operational use.
- `churn/download.py` handles acquisition; `churn/train.py` holds validated modeling, `tests/` validates contracts, `docs/` records data provenance and measured results.
- The full notebook is a readable analysis; the Python modules are the reusable implementation.